In [1]:
import numpy as np

import pyspark.sql.functions as F
from pyspark.sql.functions import udf
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, MapType, StructType, StructField, StringType, IntegerType, DoubleType

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("seminar")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
sc = spark.sparkContext

**Создание DataFrame**

Из RDD

In [3]:
def cleaning(row):
    row = row.split("\t")[:3]
    row = [float(val) for val in row]
    return row

In [4]:
ratings = sc.textFile("user_ratedmovies.dat")

first_row = ratings.first()
ratings = ratings.filter(lambda row: row != first_row).map(cleaning)

In [5]:
columns = first_row.split("\t")[:3]

In [6]:
columns

['userID', 'movieID', 'rating']

In [7]:
ratings.take(5)

[[75.0, 3.0, 1.0],
 [75.0, 32.0, 4.5],
 [75.0, 110.0, 4.0],
 [75.0, 160.0, 2.0],
 [75.0, 163.0, 4.0]]

In [8]:
df_rdd = spark.createDataFrame(ratings, columns)

In [9]:
df_rdd

DataFrame[userID: double, movieID: double, rating: double]

Можно еще вот так:

In [10]:
df_rdd = ratings.toDF(columns)

In [11]:
df_rdd

DataFrame[userID: double, movieID: double, rating: double]

Так, а если не хочу вот эти приседания с RDD, а хочу сразу из файла?

In [12]:
df = spark.read.format("csv").options(**{"sep": "\t", "header": "true"}).load("user_ratedmovies.dat")

In [13]:
df

DataFrame[userID: string, movieID: string, rating: string, date_day: string, date_month: string, date_year: string, date_hour: string, date_minute: string, date_second: string]

Все в string, так не пойдет, давайте автоматически определим тип данных

In [14]:
df = spark.read.format("csv").options(**{"sep": "\t", "header": "true", "inferSchema": "true"}).load("user_ratedmovies.dat")

In [15]:
df

DataFrame[userID: int, movieID: int, rating: double, date_day: int, date_month: int, date_year: int, date_hour: int, date_minute: int, date_second: int]

In [16]:
df.show(10)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|    75|      3|   1.0|      29|        10|     2006|       23|         17|         16|
|    75|     32|   4.5|      29|        10|     2006|       23|         23|         44|
|    75|    110|   4.0|      29|        10|     2006|       23|         30|          8|
|    75|    160|   2.0|      29|        10|     2006|       23|         16|         52|
|    75|    163|   4.0|      29|        10|     2006|       23|         29|         30|
|    75|    165|   4.5|      29|        10|     2006|       23|         25|         15|
|    75|    173|   3.5|      29|        10|     2006|       23|         17|         37|
|    75|    296|   5.0|      29|        10|     2006|       23|         24|         49|
|    75|    353|   3.5|      29|

А можно заранее сказать, какой тип данных я ожидаю?

In [17]:
schema = StructType([
    StructField("userID", IntegerType(), True),
    StructField("movieID", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("date_day", StringType(), True),
    StructField("date_month", StringType(), True),
    StructField("date_year", IntegerType(), True),
    StructField("date_hour", IntegerType(), True),
    StructField("date_minute", IntegerType(), True),
    StructField("date_second", IntegerType(), True),
])

In [18]:
df = spark.read.format("csv").options(**{"sep": "\t", "header": "true"}).schema(schema).load("user_ratedmovies.dat")

In [19]:
df.printSchema()

root
 |-- userID: integer (nullable = true)
 |-- movieID: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- date_day: string (nullable = true)
 |-- date_month: string (nullable = true)
 |-- date_year: integer (nullable = true)
 |-- date_hour: integer (nullable = true)
 |-- date_minute: integer (nullable = true)
 |-- date_second: integer (nullable = true)



In [20]:
df.show(10)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|    75|      3|   1.0|      29|        10|     2006|       23|         17|         16|
|    75|     32|   4.5|      29|        10|     2006|       23|         23|         44|
|    75|    110|   4.0|      29|        10|     2006|       23|         30|          8|
|    75|    160|   2.0|      29|        10|     2006|       23|         16|         52|
|    75|    163|   4.0|      29|        10|     2006|       23|         29|         30|
|    75|    165|   4.5|      29|        10|     2006|       23|         25|         15|
|    75|    173|   3.5|      29|        10|     2006|       23|         17|         37|
|    75|    296|   5.0|      29|        10|     2006|       23|         24|         49|
|    75|    353|   3.5|      29|

Но есть уже готовая обертка под все нужды

In [21]:
df = spark.read.csv(path="user_ratedmovies.dat", sep="\t", header=True, inferSchema=True, schema=None)

In [22]:
df.printSchema()

root
 |-- userID: integer (nullable = true)
 |-- movieID: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- date_day: integer (nullable = true)
 |-- date_month: integer (nullable = true)
 |-- date_year: integer (nullable = true)
 |-- date_hour: integer (nullable = true)
 |-- date_minute: integer (nullable = true)
 |-- date_second: integer (nullable = true)



Так, а как сохранить? Лучше быть аккуратнее с overwrite, перезапишет весь указанный путь, append будет безопаснее

In [23]:
df.write.option("header", True).mode("overwrite").parquet("write_1.parquet")

А что с партицированием?

In [24]:
df.write.option("header",True).partitionBy("date_year").mode("overwrite").parquet("write_2.parquet")

Кстати, раз уж заговорили про схемы данных, то их можно задавать интереснее, например, под группированные данные

In [25]:
structureData = [
    (("James", "", "Smith"), "36636", "M", 3100),
    (("Michael", "Rose", ""), "40288", "M", 4300),
    (("Robert", "", "Williams"), "42114", "M", 1400),
    (("Maria", "Anne", "Jones"), "39192", "F", 5500),
    (("Jen", "Mary", "Brown"), "", "F", -1),
]
structureSchema = StructType([
    StructField(
        "name",
        StructType([
            StructField("firstname", StringType(), True),
            StructField("middlename", StringType(), True),
            StructField("lastname", StringType(), True),
        ])
    ),
    StructField("id", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True),
])

df2 = spark.createDataFrame(data=structureData,schema=structureSchema)
df2.printSchema()
df2.show(truncate=False)

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)

+--------------------+-----+------+------+
|name                |id   |gender|salary|
+--------------------+-----+------+------+
|{James, , Smith}    |36636|M     |3100  |
|{Michael, Rose, }   |40288|M     |4300  |
|{Robert, , Williams}|42114|M     |1400  |
|{Maria, Anne, Jones}|39192|F     |5500  |
|{Jen, Mary, Brown}  |     |F     |-1    |
+--------------------+-----+------+------+



Со структурой можно работать и менять ее под ваши нужны

In [26]:
updatedDF = (
    df2.withColumn(
        "OtherInfo",
        F.struct(
            F.col("id").alias("identifier"),
            F.col("gender").alias("gender"),
            F.col("salary").alias("salary"),
            F.when(F.col("salary").cast(IntegerType()) < 2000, "Low")
            .when(F.col("salary").cast(IntegerType()) < 4000, "Medium")
            .otherwise("High").alias("Salary_Grade"),
        )
    )
    .drop("id","gender","salary")
)

updatedDF.printSchema()
updatedDF.show(truncate=False)

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- OtherInfo: struct (nullable = false)
 |    |-- identifier: string (nullable = true)
 |    |-- gender: string (nullable = true)
 |    |-- salary: integer (nullable = true)
 |    |-- Salary_Grade: string (nullable = false)

+--------------------+------------------------+
|name                |OtherInfo               |
+--------------------+------------------------+
|{James, , Smith}    |{36636, M, 3100, Medium}|
|{Michael, Rose, }   |{40288, M, 4300, High}  |
|{Robert, , Williams}|{42114, M, 1400, Low}   |
|{Maria, Anne, Jones}|{39192, F, 5500, High}  |
|{Jen, Mary, Brown}  |{, F, -1, Low}          |
+--------------------+------------------------+



Что мы там сделали????

1) Создали новую структуру данных OtherInfo

2) Передали туда id (переименовав столбец), gender, salary

3) Создали столбец Salary_grade из условий

4) удалили id, gender, salary из старой структуры

Есть и еще структуры данных!

In [27]:
arrayStructureSchema = StructType([
    StructField(
        "name",
        StructType([
            StructField("firstname", StringType(), True),
            StructField("middlename", StringType(), True),
            StructField("lastname", StringType(), True),
        ])),
        StructField("hobbies", ArrayType(StringType()), True),
        StructField("properties", MapType(IntegerType(), StringType()), True),
    ])

In [28]:
structureData = [
    (("James", "", "Smith"), ["car", "volleyball"], {1: "a", 4: "d"}),
    (("Michael", "Rose", ""), ["car", "football"], {2: "b"}),
    (("Robert", "", "Williams"), ["box", "music"], {3: "c"}),
  ]

In [29]:
df3 = spark.createDataFrame(data=structureData,schema=arrayStructureSchema)
df3.printSchema()
df3.show(truncate=False)

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- hobbies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- properties: map (nullable = true)
 |    |-- key: integer
 |    |-- value: string (valueContainsNull = true)

+--------------------+-----------------+----------------+
|name                |hobbies          |properties      |
+--------------------+-----------------+----------------+
|{James, , Smith}    |[car, volleyball]|{4 -> d, 1 -> a}|
|{Michael, Rose, }   |[car, football]  |{2 -> b}        |
|{Robert, , Williams}|[box, music]     |{3 -> c}        |
+--------------------+-----------------+----------------+



In [30]:
df3.select("properties").collect()

[Row(properties={4: 'd', 1: 'a'}),
 Row(properties={2: 'b'}),
 Row(properties={3: 'c'})]

**Описание данных**

Общее описание данных

In [31]:
df = spark.read.csv(path="user_ratedmovies.dat", sep="\t", header=True, inferSchema=True, schema=None)

In [32]:
df.describe().show()

+-------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|summary|            userID|           movieID|            rating|          date_day|       date_month|        date_year|         date_hour|       date_minute|       date_second|
+-------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|  count|            855598|            855598|            855598|            855598|           855598|           855598|            855598|            855598|            855598|
|   mean| 35190.83255103448| 8710.179402008887| 3.437945156487042|15.568923723524366| 6.54079485926802|2005.323717446745|12.124531614145896|29.645382527775894|29.510222090280717|
| stddev|20385.003346991376|14446.852908494207|1.0025608721610382| 8.951201361275306|3.506399425908971|2.

In [33]:
df.summary().show()

+-------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|summary|            userID|           movieID|            rating|          date_day|       date_month|        date_year|         date_hour|       date_minute|       date_second|
+-------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|  count|            855598|            855598|            855598|            855598|           855598|           855598|            855598|            855598|            855598|
|   mean| 35190.83255103448| 8710.179402008887| 3.437945156487042|15.568923723524366| 6.54079485926802|2005.323717446745|12.124531614145896|29.645382527775894|29.510222090280717|
| stddev|20385.003346991376|14446.852908494207|1.0025608721610382| 8.951201361275306|3.506399425908971|2.

Количество записей

In [34]:
df.count()

855598

Количество партиций

In [35]:
df.rdd.getNumPartitions()

2

 Менять число партиций можно, все как с rdd

In [36]:
df = df.repartition(5)

In [37]:
df.rdd.getNumPartitions()

5

In [38]:
df = df.coalesce(2)

In [39]:
df.rdd.getNumPartitions()

2

**Различные методы**

Ну теперь давайте тыкать

Удаляем дубликаты и помним, что есть actions и transformations, count заставит все сделать

In [40]:
df_without_duplicates = df.drop_duplicates()

Есть alias

In [41]:
df_without_duplicates = df.dropDuplicates()

Как удалить дубликаты по отдельным колонкам?

In [42]:
df_without_duplicates = df.drop_duplicates(["userID", "rating"])

In [43]:
df_without_duplicates.count()

18847

In [44]:
df_without_duplicates.show(10)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
| 32791|   1359|   2.0|      11|        10|     2007|       19|          8|         55|
| 18161|   3210|   3.5|      27|         2|     2006|        7|         19|         19|
| 20554|  45210|   3.0|      27|         9|     2008|        1|         42|         12|
| 22701|    861|   4.5|       7|         3|     2006|       12|         44|         33|
| 19519|   8911|   3.0|      25|         7|     2006|        0|         16|         44|
| 11930|   2732|   3.0|       2|         4|     2005|        6|         47|         25|
| 31375|   6874|   3.5|       6|         6|     2005|       12|         30|         58|
| 36619|   2858|   4.0|      30|         1|     2008|       17|         36|         27|
| 11420|   6942|   3.5|       5|

Корреляции

In [45]:
df.corr("rating", "date_day")

0.01663838844049787

In [46]:
df.corr("rating", "date_hour")

-0.012518740192686211

In [47]:
df.corr("rating", "date_year")

-0.002129926269907515

Как закинуть данные в любимый pandas?

 Самый простой вариант - встроенный метод, но он жутко медленный при существенном объеме данных. Лучше сохранить паркет и считать через pd.read_parquet()

In [48]:
import pandas as pd

In [49]:
%%timeit

pandas_df = df.toPandas()

4.1 s ± 287 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [50]:
pandas_df = df.toPandas()
pandas_df

,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second
0,38914,1732,4.0,8,2,2008,11,36,1
1,6393,2071,3.5,4,1,2006,21,28,23
2,32747,38061,4.0,5,4,2008,15,55,49
3,4352,30812,3.0,2,3,2006,22,34,12
4,37934,27674,3.5,8,10,2007,19,30,43
...,...,...,...,...,...,...,...,...,...
855593,67542,5666,4.5,11,10,2006,0,55,1
855594,51033,7346,4.0,8,10,2004,15,55,53
855595,44038,34319,3.0,10,9,2006,1,17,24
855596,63556,4720,3.5,28,10,2005,6,18,0


In [51]:
pandas_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 855598 entries, 0 to 855597
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   userID       855598 non-null  int32  
 1   movieID      855598 non-null  int32  
 2   rating       855598 non-null  float64
 3   date_day     855598 non-null  int32  
 4   date_month   855598 non-null  int32  
 5   date_year    855598 non-null  int32  
 6   date_hour    855598 non-null  int32  
 7   date_minute  855598 non-null  int32  
 8   date_second  855598 non-null  int32  
dtypes: float64(1), int32(8)
memory usage: 32.6 MB


In [52]:
%%timeit

pandas_df = pd.read_parquet("write_1.parquet")

20.2 ms ± 641 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [53]:
pandas_df = pd.read_parquet("write_1.parquet")
pandas_df

,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second
0,75,3,1.0,29,10,2006,23,17,16
1,75,32,4.5,29,10,2006,23,23,44
2,75,110,4.0,29,10,2006,23,30,8
3,75,160,2.0,29,10,2006,23,16,52
4,75,163,4.0,29,10,2006,23,29,30
...,...,...,...,...,...,...,...,...,...
855593,71534,44555,4.0,3,12,2007,3,5,38
855594,71534,46578,4.0,3,12,2007,2,56,44
855595,71534,48516,4.5,3,12,2007,2,53,46
855596,71534,61075,5.0,10,10,2008,9,56,5


In [54]:
pandas_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 855598 entries, 0 to 855597
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   userID       855598 non-null  int32  
 1   movieID      855598 non-null  int32  
 2   rating       855598 non-null  float64
 3   date_day     855598 non-null  int32  
 4   date_month   855598 non-null  int32  
 5   date_year    855598 non-null  int32  
 6   date_hour    855598 non-null  int32  
 7   date_minute  855598 non-null  int32  
 8   date_second  855598 non-null  int32  
dtypes: float64(1), int32(8)
memory usage: 32.6 MB


Как говорили на лекции, может все упасть например тут (когда делаем toPandas). Как перейти к итератору?

prefetchPartitions - подготавливать ли следующую партию данных, пока не запросили

In [55]:
iter_df = df.toLocalIterator()

In [56]:
row = iter_df.send(None)

In [57]:
row

Row(userID=38914, movieID=1732, rating=4.0, date_day=8, date_month=2, date_year=2008, date_hour=11, date_minute=36, date_second=1)

In [58]:
row.asDict()

{'userID': 38914,
 'movieID': 1732,
 'rating': 4.0,
 'date_day': 8,
 'date_month': 2,
 'date_year': 2008,
 'date_hour': 11,
 'date_minute': 36,
 'date_second': 1}

Отсюда идея: можно вытягивать данные по 1 записи и записывать в датафрейм. Долго, но зато отработает.

In [59]:
iter_df = df.toLocalIterator()

In [60]:
list_of_rows = [value for value in iter_df]
print(len(list_of_rows))

855598


In [61]:
df.columns

['userID',
 'movieID',
 'rating',
 'date_day',
 'date_month',
 'date_year',
 'date_hour',
 'date_minute',
 'date_second']

In [62]:
pandas_df = pd.DataFrame(list_of_rows, columns=df.columns)

In [63]:
%%timeit

iter_df = df.toLocalIterator()
list_of_rows = [value for value in iter_df]
pandas_df = pd.DataFrame(list_of_rows, columns=df.columns)

3.75 s ± 244 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [64]:
pandas_df

,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second
0,38914,1732,4.0,8,2,2008,11,36,1
1,6393,2071,3.5,4,1,2006,21,28,23
2,32747,38061,4.0,5,4,2008,15,55,49
3,4352,30812,3.0,2,3,2006,22,34,12
4,37934,27674,3.5,8,10,2007,19,30,43
...,...,...,...,...,...,...,...,...,...
855593,67542,5666,4.5,11,10,2006,0,55,1
855594,51033,7346,4.0,8,10,2004,15,55,53
855595,44038,34319,3.0,10,9,2006,1,17,24
855596,63556,4720,3.5,28,10,2005,6,18,0


In [65]:
pandas_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 855598 entries, 0 to 855597
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   userID       855598 non-null  int64  
 1   movieID      855598 non-null  int64  
 2   rating       855598 non-null  float64
 3   date_day     855598 non-null  int64  
 4   date_month   855598 non-null  int64  
 5   date_year    855598 non-null  int64  
 6   date_hour    855598 non-null  int64  
 7   date_minute  855598 non-null  int64  
 8   date_second  855598 non-null  int64  
dtypes: float64(1), int64(8)
memory usage: 58.7 MB


**Show**

In [66]:
df.show(10)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
| 38914|   1732|   4.0|       8|         2|     2008|       11|         36|          1|
|  6393|   2071|   3.5|       4|         1|     2006|       21|         28|         23|
| 32747|  38061|   4.0|       5|         4|     2008|       15|         55|         49|
|  4352|  30812|   3.0|       2|         3|     2006|       22|         34|         12|
| 37934|  27674|   3.5|       8|        10|     2007|       19|         30|         43|
| 30840|  34148|   3.5|       6|        10|     2005|        1|         38|         58|
|  5954|   4007|   4.0|       8|         1|     2006|       22|          5|          2|
| 26243|    223|   3.0|      19|         3|     2006|        4|          4|         30|
|  3600|   3552|   3.5|      13|

Обрезаем до 2 символов

In [67]:
df.show(10, truncate=2)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|    38|     17|    4.|       8|         2|       20|       11|         36|          1|
|    63|     20|    3.|       4|         1|       20|       21|         28|         23|
|    32|     38|    4.|       5|         4|       20|       15|         55|         49|
|    43|     30|    3.|       2|         3|       20|       22|         34|         12|
|    37|     27|    3.|       8|        10|       20|       19|         30|         43|
|    30|     34|    3.|       6|        10|       20|        1|         38|         58|
|    59|     40|    4.|       8|         1|       20|       22|          5|          2|
|    26|     22|    3.|      19|         3|       20|        4|          4|         30|
|    36|     35|    3.|      13|

вертикальное отображение

In [68]:
df.show(10, vertical=True)

-RECORD 0------------
 userID      | 38914 
 movieID     | 1732  
 rating      | 4.0   
 date_day    | 8     
 date_month  | 2     
 date_year   | 2008  
 date_hour   | 11    
 date_minute | 36    
 date_second | 1     
-RECORD 1------------
 userID      | 6393  
 movieID     | 2071  
 rating      | 3.5   
 date_day    | 4     
 date_month  | 1     
 date_year   | 2006  
 date_hour   | 21    
 date_minute | 28    
 date_second | 23    
-RECORD 2------------
 userID      | 32747 
 movieID     | 38061 
 rating      | 4.0   
 date_day    | 5     
 date_month  | 4     
 date_year   | 2008  
 date_hour   | 15    
 date_minute | 55    
 date_second | 49    
-RECORD 3------------
 userID      | 4352  
 movieID     | 30812 
 rating      | 3.0   
 date_day    | 2     
 date_month  | 3     
 date_year   | 2006  
 date_hour   | 22    
 date_minute | 34    
 date_second | 12    
-RECORD 4------------
 userID      | 37934 
 movieID     | 27674 
 rating      | 3.5   
 date_day    | 8     
 date_mont

**Select**

В PySpark метод `.select()` используется для выбора одного, нескольких столбцов по индексу, всех столбцов из списка и вложенных столбцов из фрейма данных. Метод PySpark `.select()` является функцией преобразования, поэтому она возвращает новый фрейм данных с выбранными столбцами.

In [69]:
df.columns

['userID',
 'movieID',
 'rating',
 'date_day',
 'date_month',
 'date_year',
 'date_hour',
 'date_minute',
 'date_second']

Упс, pandas-style тут не приветствуется

In [70]:
df.userID.show(5)

TypeError: 'Column' object is not callable

In [71]:
df.select("userID").show(5)

+------+
|userID|
+------+
| 16592|
| 16592|
| 16592|
| 16592|
| 16592|
+------+
only showing top 5 rows


Куча вариантов, выбирайте любой

In [72]:
df.select("userID", "rating").show(5)

+------+------+
|userID|rating|
+------+------+
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
+------+------+
only showing top 5 rows


In [73]:
df.select(["userID", "rating"]).show(5)

+------+------+
|userID|rating|
+------+------+
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
+------+------+
only showing top 5 rows


In [74]:
df.select(df.userID,df.rating).show(5)

+------+------+
|userID|rating|
+------+------+
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
+------+------+
only showing top 5 rows


In [75]:
df.select(df["userID"], df["rating"]).show(5)

+------+------+
|userID|rating|
+------+------+
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
+------+------+
only showing top 5 rows


In [76]:
df.select(F.col("userID"), F.col("rating")).show(5)

+------+------+
|userID|rating|
+------+------+
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
| 13216|   4.5|
+------+------+
only showing top 5 rows


можно налету переименовать столбец

In [77]:
df.select(df.userID, df.rating.alias("mark")).show(5)

+------+----+
|userID|mark|
+------+----+
| 13216| 4.5|
| 13216| 4.5|
| 13216| 4.5|
| 13216| 4.5|
| 13216| 4.5|
+------+----+
only showing top 5 rows


In [78]:
df.select(df.colRegex("`d+.*y`")).show(5)

+--------+
|date_day|
+--------+
|      19|
|      19|
|      19|
|      19|
|      19|
+--------+
only showing top 5 rows


примеры с другим датафреймом, где структура сложнее

In [79]:
structureData = [
    (("James", "", "Smith"), "36636", "M", 3100),
    (("Michael", "Rose", ""), "40288", "M", 4300),
    (("Robert", "", "Williams"), "42114", "M", 1400),
    (("Maria", "Anne", "Jones"), "39192", "F", 5500),
    (("Jen", "Mary", "Brown"), "", "F", -1),
  ]
structureSchema = StructType([
    StructField(
        "name",
        StructType([
            StructField("firstname", StringType(), True),
            StructField("middlename", StringType(), True),
            StructField("lastname", StringType(), True),
        ]),
    ),
    StructField("id", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True),
])

df2 = spark.createDataFrame(data=structureData,schema=structureSchema)
df2.printSchema()
df2.show(truncate=False)

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)

+--------------------+-----+------+------+
|name                |id   |gender|salary|
+--------------------+-----+------+------+
|{James, , Smith}    |36636|M     |3100  |
|{Michael, Rose, }   |40288|M     |4300  |
|{Robert, , Williams}|42114|M     |1400  |
|{Maria, Anne, Jones}|39192|F     |5500  |
|{Jen, Mary, Brown}  |     |F     |-1    |
+--------------------+-----+------+------+



In [80]:
df2.select("name").show(5)

+--------------------+
|                name|
+--------------------+
|    {James, , Smith}|
|   {Michael, Rose, }|
|{Robert, , Williams}|
|{Maria, Anne, Jones}|
|  {Jen, Mary, Brown}|
+--------------------+



In [81]:
df2.select("name.lastname").show(5)

+--------+
|lastname|
+--------+
|   Smith|
|        |
|Williams|
|   Jones|
|   Brown|
+--------+



In [82]:
df2.select("name.firstname", "name.lastname").show(5)

+---------+--------+
|firstname|lastname|
+---------+--------+
|    James|   Smith|
|  Michael|        |
|   Robert|Williams|
|    Maria|   Jones|
|      Jen|   Brown|
+---------+--------+



**withColumn**

PySpark `.withColumn()` - это функция преобразования (transform), которая используется для изменения значения, преобразования типа данных существующего столбца, создания нового столбца и многого другого. Поговорим о часто используемых операциях со столбцами данных PySpark, используя примеры.

In [83]:
df.printSchema()

root
 |-- userID: integer (nullable = true)
 |-- movieID: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- date_day: integer (nullable = true)
 |-- date_month: integer (nullable = true)
 |-- date_year: integer (nullable = true)
 |-- date_hour: integer (nullable = true)
 |-- date_minute: integer (nullable = true)
 |-- date_second: integer (nullable = true)



Меняем тип данных

In [84]:
df.withColumn("date_month", F.col("date_month").cast("String")).printSchema()

root
 |-- userID: integer (nullable = true)
 |-- movieID: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- date_day: integer (nullable = true)
 |-- date_month: string (nullable = true)
 |-- date_year: integer (nullable = true)
 |-- date_hour: integer (nullable = true)
 |-- date_minute: integer (nullable = true)
 |-- date_second: integer (nullable = true)



Модифицировать столбец/создать новый

In [85]:
df.withColumn("rating_x_10",F.col("rating") * 10).show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|rating_x_10|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+-----------+
| 38914|   1732|   4.0|       8|         2|     2008|       11|         36|          1|       40.0|
|  6393|   2071|   3.5|       4|         1|     2006|       21|         28|         23|       35.0|
| 32747|  38061|   4.0|       5|         4|     2008|       15|         55|         49|       40.0|
|  4352|  30812|   3.0|       2|         3|     2006|       22|         34|         12|       30.0|
| 37934|  27674|   3.5|       8|        10|     2007|       19|         30|         43|       35.0|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+-----------+
only showing top 5 rows


Делаем 2 константных столбца

In [86]:
df.withColumn("fix_1", F.lit(1)).withColumn("fix_2", F.lit(2)).show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+-----+-----+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|fix_1|fix_2|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+-----+-----+
| 38914|   1732|   4.0|       8|         2|     2008|       11|         36|          1|    1|    2|
|  6393|   2071|   3.5|       4|         1|     2006|       21|         28|         23|    1|    2|
| 32747|  38061|   4.0|       5|         4|     2008|       15|         55|         49|    1|    2|
|  4352|  30812|   3.0|       2|         3|     2006|       22|         34|         12|    1|    2|
| 37934|  27674|   3.5|       8|        10|     2007|       19|         30|         43|    1|    2|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+-----+-----+
only showing top 5 rows


**withColumnsRenamed**

Предыдущий вариант не давал возможности переименовать столбцы, это можно сделать иначе

In [87]:
df.withColumnRenamed("rating", "mark").show(5)

+------+-------+----+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|mark|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+----+--------+----------+---------+---------+-----------+-----------+
| 38914|   1732| 4.0|       8|         2|     2008|       11|         36|          1|
|  6393|   2071| 3.5|       4|         1|     2006|       21|         28|         23|
| 32747|  38061| 4.0|       5|         4|     2008|       15|         55|         49|
|  4352|  30812| 3.0|       2|         3|     2006|       22|         34|         12|
| 37934|  27674| 3.5|       8|        10|     2007|       19|         30|         43|
+------+-------+----+--------+----------+---------+---------+-----------+-----------+
only showing top 5 rows


**filter (where) и иные филтрации**

Функция PySpark `.filter()` используется для фильтрации строк из RDD / DataFrame на основе заданного условия или выражения SQL, вы также можете использовать предложение `.where()` вместо `.filter()` обе эти функции работают аналогично.

одно условие

In [88]:
df.filter(df.rating == 5.0).show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
| 35105|  41569|   5.0|      31|        12|     2005|        7|         31|         38|
| 30977|  45950|   5.0|      10|         7|     2006|        8|         16|          0|
| 36969|   1251|   5.0|       7|         5|     2006|       22|         35|         15|
|  7252|    296|   5.0|      29|        11|     2006|       19|         39|         34|
|  5917|   3196|   5.0|      12|         1|     2005|       10|         24|          6|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
only showing top 5 rows


In [89]:
df.filter(~(df.rating == 5.0)).show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
| 38914|   1732|   4.0|       8|         2|     2008|       11|         36|          1|
|  6393|   2071|   3.5|       4|         1|     2006|       21|         28|         23|
| 32747|  38061|   4.0|       5|         4|     2008|       15|         55|         49|
|  4352|  30812|   3.0|       2|         3|     2006|       22|         34|         12|
| 37934|  27674|   3.5|       8|        10|     2007|       19|         30|         43|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
only showing top 5 rows


In [90]:
df.filter("rating = 5").show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
| 35105|  41569|   5.0|      31|        12|     2005|        7|         31|         38|
| 30977|  45950|   5.0|      10|         7|     2006|        8|         16|          0|
| 36969|   1251|   5.0|       7|         5|     2006|       22|         35|         15|
|  7252|    296|   5.0|      29|        11|     2006|       19|         39|         34|
|  5917|   3196|   5.0|      12|         1|     2005|       10|         24|          6|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
only showing top 5 rows


Несколько условий

In [91]:
df.filter((df.rating == 5.0) & (df.date_year == 2006)).show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
| 36969|   1251|   5.0|       7|         5|     2006|       22|         35|         15|
| 22278|   1732|   5.0|       5|         3|     2006|       16|         27|          7|
|  2853|    296|   5.0|      10|        11|     2006|       13|         52|         53|
|  4895|  27728|   5.0|       1|         1|     2006|        4|         15|          2|
| 19532|   2959|   5.0|      20|         2|     2006|        1|         51|         40|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
only showing top 5 rows


In [92]:
df.filter("(rating = 5.0) and (date_year = 2006)").show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
| 36969|   1251|   5.0|       7|         5|     2006|       22|         35|         15|
| 22278|   1732|   5.0|       5|         3|     2006|       16|         27|          7|
|  2853|    296|   5.0|      10|        11|     2006|       13|         52|         53|
|  4895|  27728|   5.0|       1|         1|     2006|        4|         15|          2|
| 19532|   2959|   5.0|      20|         2|     2006|        1|         51|         40|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
only showing top 5 rows


In [93]:
df.filter("(rating = 5.0) and (userID between 70 and 80)").show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|    78|   1214|   5.0|       7|         5|     2004|       23|          6|         34|
|    78|     32|   5.0|       7|         5|     2004|       23|         32|         18|
|    78|   2502|   5.0|       7|         5|     2004|       23|         38|         29|
|    78|    750|   5.0|       7|         5|     2004|       23|          3|         45|
|    78|   1232|   5.0|       7|         5|     2004|       23|         29|         49|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
only showing top 5 rows


фильтр по списку значений из list

In [94]:
years = [2006, 2007]

In [95]:
df.filter(df.date_year.isin(years)).show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|  6393|   2071|   3.5|       4|         1|     2006|       21|         28|         23|
| 37934|  27674|   3.5|       8|        10|     2007|       19|         30|         43|
| 25038|   3250|   3.0|      30|        12|     2007|        8|         34|         29|
| 26243|    223|   3.0|      19|         3|     2006|        4|          4|         30|
| 15311|   1411|   3.0|      22|        12|     2007|       22|         42|          4|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
only showing top 5 rows


 проверим

In [96]:
df.filter(df.date_year.isin(years)).select("date_year").dropDuplicates().collect()

[Row(date_year=2007), Row(date_year=2006)]

создадим игрушечный датайфрейм для текстовых столбцов

In [97]:
data2 = [
    (2, "Michael Rose"),
    (3, "Robert Williams"),
    (4, "Rames Rose"),
    (5, "Rames Black"),
    (6, "Albus Torch"),
    (7, "Fred Tf"),
]
df2 = spark.createDataFrame(data2, ["id", "name"])

In [98]:
df2.show()

+---+---------------+
| id|           name|
+---+---------------+
|  2|   Michael Rose|
|  3|Robert Williams|
|  4|     Rames Rose|
|  5|    Rames Black|
|  6|    Albus Torch|
|  7|        Fred Tf|
+---+---------------+



In [99]:
df2.filter('name like "R%"').show()

+---+---------------+
| id|           name|
+---+---------------+
|  3|Robert Williams|
|  4|     Rames Rose|
|  5|    Rames Black|
+---+---------------+



In [100]:
df2.filter(df2.name.startswith("R")).show()

+---+---------------+
| id|           name|
+---+---------------+
|  3|Robert Williams|
|  4|     Rames Rose|
|  5|    Rames Black|
+---+---------------+



In [101]:
df2.filter(df2.name.endswith("Tf")).show()

+---+-------+
| id|   name|
+---+-------+
|  7|Fred Tf|
+---+-------+



In [102]:
df2.filter(df2.name.contains("Wil")).show()

+---+---------------+
| id|           name|
+---+---------------+
|  3|Robert Williams|
+---+---------------+



Бывает, что у нас внутри датафрейма есть массив и с ним что-то хочется сделать

In [103]:
arrayStructureSchema = StructType([
    StructField(
        "name",
        StructType([
            StructField("firstname", StringType(), True),
            StructField("middlename", StringType(), True),
            StructField("lastname", StringType(), True),
        ])
    ),
    StructField("hobbies", ArrayType(StringType()), True),
    StructField("properties", MapType(IntegerType(), StringType()), True),
])

structureData = [
    (("James", "", "Smith"), ["car", "volleyball"], {1: "a", 4: "d"}),
    (("Michael", "Rose", ""), ["car", "football"], {2: "b"}),
    (("Robert", "", "Williams"), ["box", "music"], {3: "c"}),
]

df3 = spark.createDataFrame(data=structureData,schema=arrayStructureSchema)
df3.printSchema()
df3.show(truncate=False)

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- hobbies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- properties: map (nullable = true)
 |    |-- key: integer
 |    |-- value: string (valueContainsNull = true)

+--------------------+-----------------+----------------+
|name                |hobbies          |properties      |
+--------------------+-----------------+----------------+
|{James, , Smith}    |[car, volleyball]|{4 -> d, 1 -> a}|
|{Michael, Rose, }   |[car, football]  |{2 -> b}        |
|{Robert, , Williams}|[box, music]     |{3 -> c}        |
+--------------------+-----------------+----------------+



In [104]:
df3.filter(F.array_contains(df3.hobbies,"football")).show()

+-----------------+---------------+----------+
|             name|        hobbies|properties|
+-----------------+---------------+----------+
|{Michael, Rose, }|[car, football]|  {2 -> b}|
+-----------------+---------------+----------+



**Сортировка**

сделаем еще фильтрацию, чтобы увидеть результат (orderBy тут аналог)

In [105]:
df.filter(df.userID == 75).sort(df.date_minute, df.rating.desc()).show(20)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+
|    75|   1304|   2.5|      29|        10|     2006|       23|         16|         56|
|    75|   2011|   2.0|      29|        10|     2006|       23|         16|         39|
|    75|    420|   2.0|      29|        10|     2006|       23|         16|         42|
|    75|    160|   2.0|      29|        10|     2006|       23|         16|         52|
|    75|    832|   4.5|      29|        10|     2006|       23|         17|         49|
|    75|   2700|   4.5|      29|        10|     2006|       23|         17|         52|
|    75|   1485|   4.0|      29|        10|     2006|       23|         17|         46|
|    75|   1374|   4.0|      29|        10|     2006|       23|         17|         20|
|    75|    173|   3.5|      29|

**groupby**

Когда мы выполняем `.groupBy()` в PySpark DataFrame, он возвращает объект `GroupedData`, который содержит следующие агрегатные функции:

`min()`, `max()`, `mean()`, `count()`, `sum()`, `avg()`, `agg()`, `pivot()`

In [106]:
df.groupby("date_year").mean("rating").collect()

[Row(date_year=2003, avg(rating)=3.4279154852282083),
 Row(date_year=2007, avg(rating)=3.4367384383270894),
 Row(date_year=2006, avg(rating)=3.4294610550371356),
 Row(date_year=1997, avg(rating)=3.571018651362984),
 Row(date_year=2004, avg(rating)=3.3916603359500295),
 Row(date_year=1998, avg(rating)=3.5080568720379146),
 Row(date_year=2009, avg(rating)=3.369243156199678),
 Row(date_year=2001, avg(rating)=3.4490124195720484),
 Row(date_year=2005, avg(rating)=3.395790002433071),
 Row(date_year=2000, avg(rating)=3.5767733835530446),
 Row(date_year=2008, avg(rating)=3.486752231109465),
 Row(date_year=1999, avg(rating)=3.4925797284496367),
 Row(date_year=2002, avg(rating)=3.4140127388535033)]

мы уже умеем применять разные методы

In [107]:
df.groupby("date_year").mean("rating").sort("date_year").collect()

[Row(date_year=1997, avg(rating)=3.571018651362984),
 Row(date_year=1998, avg(rating)=3.5080568720379146),
 Row(date_year=1999, avg(rating)=3.4925797284496367),
 Row(date_year=2000, avg(rating)=3.5767733835530446),
 Row(date_year=2001, avg(rating)=3.4490124195720484),
 Row(date_year=2002, avg(rating)=3.4140127388535033),
 Row(date_year=2003, avg(rating)=3.4279154852282083),
 Row(date_year=2004, avg(rating)=3.3916603359500295),
 Row(date_year=2005, avg(rating)=3.395790002433071),
 Row(date_year=2006, avg(rating)=3.4294610550371356),
 Row(date_year=2007, avg(rating)=3.4367384383270894),
 Row(date_year=2008, avg(rating)=3.486752231109465),
 Row(date_year=2009, avg(rating)=3.369243156199678)]

In [108]:
df.filter(df.rating <= 2).groupby("date_year").count().withColumnRenamed("count", "number").sort("date_year").collect()

[Row(date_year=1997, number=98),
 Row(date_year=1998, number=321),
 Row(date_year=1999, number=2270),
 Row(date_year=2000, number=4732),
 Row(date_year=2001, number=4793),
 Row(date_year=2002, number=5791),
 Row(date_year=2003, number=7595),
 Row(date_year=2004, number=10205),
 Row(date_year=2005, number=15982),
 Row(date_year=2006, number=23463),
 Row(date_year=2007, number=19792),
 Row(date_year=2008, number=15128),
 Row(date_year=2009, number=369)]

несколько колонок

In [109]:
df.groupBy("date_year", "date_month").mean("rating", "userID").sort("date_year", "date_month").show()

+---------+----------+------------------+------------------+
|date_year|date_month|       avg(rating)|       avg(userID)|
+---------+----------+------------------+------------------+
|     1997|         9|3.7094972067039107| 35144.61452513967|
|     1997|        10|3.7762237762237763|34880.769230769234|
|     1997|        11| 3.360655737704918| 9451.180327868853|
|     1997|        12| 3.549618320610687| 20251.01526717557|
|     1998|         1|          3.765625|      22209.515625|
|     1998|         2|               3.3|           31662.4|
|     1998|         3| 3.238095238095238| 19582.04761904762|
|     1998|         4| 3.908333333333333|31813.866666666665|
|     1998|         5| 3.652054794520548|20393.520547945205|
|     1998|         6|             3.225|         27092.625|
|     1998|         7|3.5547550432276656|38376.249279538904|
|     1998|         8| 3.642857142857143|41151.357142857145|
|     1998|         9|3.1881533101045294| 46104.74912891986|
|     1998|        10|3.

Для того, чтобы делать несколько разных агрегаций и еще менять сразу имя столбца нужно немного изменить синтаксис

И стоит помнить, что персентили тут считаются приближенно

In [110]:
(
    df.groupBy("date_year")
    .agg(
        F.min("rating").alias("min_rating"),
        F.mean("rating").alias("mean_rating"),
        F.max("rating").alias("max_rating"),
        F.percentile_approx("rating", 0.5).alias("median"),
    )
    .show()
)

+---------+----------+------------------+----------+------+
|date_year|min_rating|       mean_rating|max_rating|median|
+---------+----------+------------------+----------+------+
|     2003|       0.5|3.4279154852282083|       5.0|   3.5|
|     2007|       0.5|3.4367384383270894|       5.0|   3.5|
|     2006|       0.5|3.4294610550371356|       5.0|   3.5|
|     1997|       1.0| 3.571018651362984|       5.0|   4.0|
|     2004|       0.5|3.3916603359500295|       5.0|   3.5|
|     1998|       1.0|3.5080568720379146|       5.0|   4.0|
|     2009|       0.5| 3.369243156199678|       5.0|   3.5|
|     2001|       1.0|3.4490124195720484|       5.0|   4.0|
|     2005|       0.5| 3.395790002433071|       5.0|   3.5|
|     2000|       1.0|3.5767733835530446|       5.0|   4.0|
|     2008|       0.5| 3.486752231109465|       5.0|   3.5|
|     1999|       1.0|3.4925797284496367|       5.0|   4.0|
|     2002|       1.0|3.4140127388535033|       5.0|   4.0|
+---------+----------+------------------

Еще можно сделать pivot

In [111]:
df.groupBy("date_year").pivot("date_month").mean("rating").show(5)

+---------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|date_year|                 1|                 2|                 3|                 4|                 5|                 6|                 7|                 8|                 9|                10|                11|                12|
+---------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|     2003|3.4720922980979108| 3.333741687084354|3.4834824501032347|3.4579510703363914| 3.366686911630732|  3.55358230160341|3.5085277150739906| 3.447642637512812| 3.579355840124175| 3.165437788018433| 3.446676231468197| 3.440287081339713|
|     2007| 3.520191839855425| 3.3341516

**Join's**

Куда же без них. Что есть: INNER, LEFT OUTER, RIGHT OUTER, LEFT ANTI, LEFT SEMI, CROSS, SELF JOIN

Благодаря оптимизации в датафреймах уже все хорошо работает, спасибо catalist, но чудеса не вечны и плохой код/незнание данных все равно даст о себе знать

Сделаем для себя несколько таблиц, чтобы можно было экспериментировать

In [112]:
df_mean_user_rating = df.groupBy("userID").mean("rating").withColumnRenamed("avg(rating)", "avg_rating_all")
df_mean_user_rating_year = df.groupby("userID", "date_year").mean("rating").withColumnRenamed("avg(rating)", "avg_rating_year")

In [113]:
df_mean_user_rating.printSchema()

df_mean_user_rating_year.printSchema()

root
 |-- userID: integer (nullable = true)
 |-- avg_rating_all: double (nullable = true)

root
 |-- userID: integer (nullable = true)
 |-- date_year: integer (nullable = true)
 |-- avg_rating_year: double (nullable = true)



И давайте все в 1 блоке кода, чтобы не растягивать

In [114]:
(
    df
    .join(
        df_mean_user_rating,
        on=df.userID == df_mean_user_rating.userID,
        how="inner",
    )
    .join(
        df_mean_user_rating_year,
        on=[df.userID == df_mean_user_rating_year.userID, df.date_year == df_mean_user_rating_year.date_year],
        how="inner",
    )
)

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `userID` is ambiguous, could be: [`userID`, `userID`]. SQLSTATE: 42704

Надо удалить дублирующиеся столбцы

In [115]:
res_join = (
    df.alias("t")
    .join(df_mean_user_rating.alias("t1"), on=F.col("t.userID") == F.col("t1.userID"), how="inner")
    .drop(F.col("t1.userID"))
    .join(
        df_mean_user_rating_year.alias("t2"),
        on=[F.col("t.userID") == F.col("t2.userID"), F.col("t.date_year") == F.col("t2.date_year")],
        how="inner",
    )
    .drop(F.col("t2.userID"))
    .drop(F.col("t2.date_year"))
)

In [116]:
res_join.show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+------------------+-----------------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|    avg_rating_all|  avg_rating_year|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+------------------+-----------------+
| 67068|   4326|   4.0|      11|         6|     2003|        2|         35|         14|3.2815934065934065|3.358974358974359|
| 67068|   6448|   3.5|      11|         6|     2003|        2|         40|         46|3.2815934065934065|3.358974358974359|
| 67068|   5016|   4.0|       3|         2|     2003|       22|         22|         58|3.2815934065934065|3.358974358974359|
| 67068|   3735|   3.5|      15|        11|     2003|       20|         20|         33|3.2815934065934065|3.358974358974359|
| 67068|   1346|   4.0|      25|         8|     2003|        0|         35|         59|3.2815934065934065|3.358974358974359|


**union и unionAll**

Используются для объединения датафреймов с одинаковой структурой, используется union, так как unionAll с версии 2.0.0 более не используется

In [117]:
df1 = df.filter(df.date_year == 2006)
df2 = df.filter(df.date_year != 2006)
union_df = df1.union(df2)

print(df.count(), union_df.count())

855598 855598


Desclaimer: все по sql, надеюсь, помнят разницу между union и union all, когда union убирает дубликаты. Так вот pyspark ничего не удаляет, убрать дубликаты можно только через drop_duplicates, distinct

Также при union pyspark делает объединение по столбцам as is, не пытаясь понять, что в одном датафрейме нужный стобец на 1 позиции, а в другом он на 5. Для этого с версии 3.1 есть замечательный метод unionByName

**UDF - user defined functions**

из курса про rdd помним про map, тут тоже можно перегнать все в rdd и делать map, но можно и через udf. Стоит отметить, что при этом мы теряем возможность оптимизации и произодительность в dataframe, так как udf - black box для спарка.

Но зато эти функции переиспользуемы и их можно применять в sql запросах, как те же udf в oracle

In [118]:
def udf_example(rating):
    return rating * 20

In [119]:
my_udf = udf(lambda x: udf_example(x), DoubleType())

In [120]:
df.select(["userID", "movieID", "rating"]).withColumn("rating_100", my_udf(F.col("rating"))).show(5)

+------+-------+------+----------+
|userID|movieID|rating|rating_100|
+------+-------+------+----------+
| 24515|  54272|   4.0|      80.0|
|  3190|   6303|   3.5|      70.0|
|  8881|   2942|   3.0|      60.0|
| 30687|   4226|   4.0|      80.0|
| 34368|   1213|   4.0|      80.0|
+------+-------+------+----------+
only showing top 5 rows


Для тех, кто любит декораторы

In [121]:
@udf(returnType=DoubleType())
def udf_example_decorator(rating):
    return rating * 20

In [122]:
df.select(["userID", "movieID", "rating"]).withColumn("rating_100", udf_example_decorator(F.col("rating"))).show(5)

+------+-------+------+----------+
|userID|movieID|rating|rating_100|
+------+-------+------+----------+
| 24515|  54272|   4.0|      80.0|
|  3190|   6303|   3.5|      70.0|
|  8881|   2942|   3.0|      60.0|
| 30687|   4226|   4.0|      80.0|
| 34368|   1213|   4.0|      80.0|
+------+-------+------+----------+
only showing top 5 rows


Зарегистрируем функцию для будущих примеров с sql

In [123]:
spark.udf.register("udf_example_decorator", udf_example_decorator)

**SQL**

Ну раз уж пошла такая тема, давайте рассмотрим, как можно сделать все при помощи любимого SQL

можно делать TempView и GlodalTempView, отличие в том, что обычный view будет жить, пока жива сессия спрака, а глобальная, пока жив sparkcontext

In [124]:
df.createOrReplaceTempView("df")

In [125]:
query = """
SELECT
    userID,
    movieID,
    rating,
    udf_example_decorator(rating) AS rating_100
FROM df
"""
spark.sql(query).show(5)

+------+-------+------+----------+
|userID|movieID|rating|rating_100|
+------+-------+------+----------+
| 24515|  54272|   4.0|      80.0|
|  3190|   6303|   3.5|      70.0|
|  8881|   2942|   3.0|      60.0|
| 30687|   4226|   4.0|      80.0|
| 34368|   1213|   4.0|      80.0|
+------+-------+------+----------+
only showing top 5 rows


Ну и наш join

In [126]:
df_mean_user_rating.createOrReplaceTempView("df_mean_user_rating")
df_mean_user_rating_year.createOrReplaceTempView("df_mean_user_rating_year")

In [127]:
query = """
SELECT
    t.*,
    t1.avg_rating_all,
    t2.avg_rating_year
FROM df AS t, df_mean_user_rating AS t1, df_mean_user_rating_year AS t2
WHERE
    t.userID = t1.userID
    AND t.userID = t2.userID
    AND t.date_year = t2.date_year
"""
spark.sql(query).show(5)

+------+-------+------+--------+----------+---------+---------+-----------+-----------+------------------+------------------+
|userID|movieID|rating|date_day|date_month|date_year|date_hour|date_minute|date_second|    avg_rating_all|   avg_rating_year|
+------+-------+------+--------+----------+---------+---------+-----------+-----------+------------------+------------------+
|  5954|  61250|   1.0|      30|        11|     2008|       18|         20|          0|3.7425213675213675|3.6219512195121952|
| 37782|  44191|   4.0|      22|        10|     2006|       21|         14|         36|3.4693877551020407| 3.611111111111111|
| 23712|   4979|   5.0|      27|         3|     2008|        2|         45|         24|3.7753303964757707|3.8162393162393164|
| 32656|   1213|   4.0|      12|         1|     2002|        5|         17|         15| 3.939914163090129|4.0227272727272725|
| 23712|   4642|   5.0|      27|         3|     2008|        2|         45|         30|3.7753303964757707|3.8162393162

**fill() и fillna()**

Оба метода идентичны, заполняют пропуски

In [128]:
data2 = [
    (2, "Michael Rose"),
    (3, "Robert Williams"),
    (4, "Rames Rose"),
    (5, None),
    (6, None),
    (None, "Fred Tf"),
]
df2 = spark.createDataFrame(data2, ["id", "name"])

А где пропуски?

In [129]:
df2.select([F.count(F.when(F.isnan(F.col(c).try_cast("double")) | F.col(c).isNull(), c)).alias(c) for c in df2.columns]).show()

+---+----+
| id|name|
+---+----+
|  1|   2|
+---+----+



In [130]:
df2.show()

+----+---------------+
|  id|           name|
+----+---------------+
|   2|   Michael Rose|
|   3|Robert Williams|
|   4|     Rames Rose|
|   5|           NULL|
|   6|           NULL|
|NULL|        Fred Tf|
+----+---------------+



In [131]:
df2.fillna({"id": 0}).show()

+---+---------------+
| id|           name|
+---+---------------+
|  2|   Michael Rose|
|  3|Robert Williams|
|  4|     Rames Rose|
|  5|           NULL|
|  6|           NULL|
|  0|        Fred Tf|
+---+---------------+



In [132]:
df2.fillna({"id": 0, "name": "Unknown"}).show()

+---+---------------+
| id|           name|
+---+---------------+
|  2|   Michael Rose|
|  3|Robert Williams|
|  4|     Rames Rose|
|  5|        Unknown|
|  6|        Unknown|
|  0|        Fred Tf|
+---+---------------+



Аналогично

In [133]:
df2.na.fill({"id": 0, "name": "Unknown"}).show()

+---+---------------+
| id|           name|
+---+---------------+
|  2|   Michael Rose|
|  3|Robert Williams|
|  4|     Rames Rose|
|  5|        Unknown|
|  6|        Unknown|
|  0|        Fred Tf|
+---+---------------+



Немного поговорим о том, как делаеть подвыборки

**sample и sampleBy**

Не забываем про возможную некоторую недетерминированность

In [134]:
trans_data = spark.read.parquet("spark_transactions.parquet")
trans_data = trans_data.withColumn("target", F.when(F.col("IsFraud") == "No", 0).otherwise(1))

In [135]:
trans_data.count()

24386900

In [136]:
trans_data.select(F.mean(F.col("target"))).show()

+--------------------+
|         avg(target)|
+--------------------+
|0.001220204290008...|
+--------------------+



In [137]:
trans_data.select("target").groupBy("target").count().show()

+------+--------+
|target|   count|
+------+--------+
|     1|   29757|
|     0|24357143|
+------+--------+



In [138]:
trans_data_simple = trans_data.sample(withReplacement=False, fraction=0.1, seed=3)
print(trans_data_simple.count())
trans_data_simple.select(F.mean(F.col("target"))).show()

2439262
+-------------------+
|        avg(target)|
+-------------------+
|0.00120897222192614|
+-------------------+



Стратификация

Тут важно понимать, что это не scikit-learn и стратификация предполагает, что вы по какому-то полю можете выбрать определенную долю наблюдений по его значениям

In [139]:
trans_data.sampleBy(F.col("target"), fractions={1: 1.0}, seed=0).select("target").groupBy("target").count().show()

+------+-----+
|target|count|
+------+-----+
|     1|29757|
+------+-----+



Не указали какой-то ключ - его доля будет 0

In [140]:
trans_data.sampleBy(F.col("target"), fractions={1: 1.0, 0: 0.1}, seed=0).select("target").groupBy("target").count().show()

+------+-------+
|target|  count|
+------+-------+
|     1|  29757|
|     0|2436333|
+------+-------+



Оконные функции **F.func().over(Window.partitionBy().orderBy())**

In [141]:
from pyspark.sql.window import Window

In [142]:
(
    trans_data_simple
    .select("User", "Card", "Year", "Month", "Day")
    .withColumn("rn_1", F.row_number().over(Window.partitionBy("User", "Card").orderBy(F.col("Year").asc())))
    .withColumn("rn_2", F.row_number().over(Window.partitionBy("User", "Card").orderBy(F.col("Year").asc(), F.col("Month").desc())))
    .withColumn("mean", F.mean("Day").over(Window.partitionBy("User", "Card", "Month")))
    .withColumn("lag", F.lag("Day").over(Window.partitionBy("User", "Card", "Month").orderBy(F.col("Year").asc(), F.col("Month").desc(), F.col("Day").desc())))
    .show()
)

+----+----+----+-----+---+----+----+------------------+----+
|User|Card|Year|Month|Day|rn_1|rn_2|              mean| lag|
+----+----+----+-----+---+----+----+------------------+----+
|   0|   1|2015|    1| 24|  19|  37|              13.6|NULL|
|   0|   1|2015|    1| 16|  18|  36|              13.6|  24|
|   0|   1|2016|    1|  8|  41|  57|              13.6|  16|
|   0|   1|2016|    1|  5|  40|  56|              13.6|   8|
|   0|   1|2016|    1|  3|  39|  55|              13.6|   5|
|   0|   1|2016|    1|  1|  38|  54|              13.6|   3|
|   0|   1|2018|    1| 27|  74|  88|              13.6|   1|
|   0|   1|2018|    1|  5|  73|  87|              13.6|  27|
|   0|   1|2020|    1| 26| 102| 105|              13.6|   5|
|   0|   1|2020|    1| 21| 101| 104|              13.6|  26|
|   0|   1|2015|    2| 15|  20|  35|15.666666666666666|NULL|
|   0|   1|2016|    2|  7|  42|  53|15.666666666666666|  15|
|   0|   1|2018|    2| 23|  76|  86|15.666666666666666|   7|
|   0|   1|2018|    2|  

Сборка последовательностей через **collect_list()**

In [143]:
trans_data.printSchema()

root
 |-- User: long (nullable = true)
 |-- Card: long (nullable = true)
 |-- Year: long (nullable = true)
 |-- Month: long (nullable = true)
 |-- Day: long (nullable = true)
 |-- Time: string (nullable = true)
 |-- Amount: string (nullable = true)
 |-- UseChip: string (nullable = true)
 |-- MerchantCity: string (nullable = true)
 |-- MerchantState: string (nullable = true)
 |-- Zip: double (nullable = true)
 |-- MCC: long (nullable = true)
 |-- IsFraud: string (nullable = true)
 |-- target: integer (nullable = false)



In [144]:
trans_data_simple_seq = trans_data_simple.select("User", "Card", F.struct("Amount", "Day").alias("sequence"))

In [145]:
trans_data_simple_seq.show()

+----+----+-------------+
|User|Card|     sequence|
+----+----+-------------+
|   0|   0|{$162.39, 11}|
|   0|   0|{$159.72, 17}|
|   0|   0| {$92.30, 18}|
|   0|   0|{$168.11, 19}|
|   0|   0|{$119.69, 20}|
|   0|   0| {$48.34, 21}|
|   0|   0|{$141.70, 28}|
|   0|   0| {$40.58, 29}|
|   0|   0|  {$16.87, 5}|
|   0|   0|  {$1.84, 18}|
|   0|   0| {$57.00, 21}|
|   0|   0| {$66.00, 24}|
|   0|   0| {$35.82, 25}|
|   0|   0| {$77.85, 30}|
|   0|   0| {$96.00, 31}|
|   0|   0|  {$30.50, 2}|
|   0|   0|  {$75.72, 4}|
|   0|   0| {$122.64, 8}|
|   0|   0|{$127.33, 18}|
|   0|   0| {$83.17, 20}|
+----+----+-------------+
only showing top 20 rows


In [146]:
trans_data_simple_seq = trans_data_simple_seq.groupBy("User", "Card").agg(F.collect_list("sequence").alias("sequence"))

In [147]:
trans_data_simple_seq.show(truncate=50)

+----+----+--------------------------------------------------+
|User|Card|                                          sequence|
+----+----+--------------------------------------------------+
|   0|   1|[{$20.61, 29}, {$42.80, 6}, {$13.49, 6}, {$149....|
|   1|   0|                      [{$60.86, 18}, {$29.18, 18}]|
|   1|   1|[{$120.00, 2}, {$24.63, 10}, {$67.21, 13}, {$14...|
|   1|   3|[{$12.90, 25}, {$2.72, 14}, {$55.14, 3}, {$96.0...|
|   1|   4|[{$90.43, 8}, {$10.77, 16}, {$0.67, 26}, {$14.1...|
|   3|   0|[{$141.16, 9}, {$340.00, 10}, {$400.00, 15}, {$...|
|   4|   0|[{$42.88, 2}, {$464.35, 12}, {$103.37, 14}, {$1...|
|   5|   1|[{$113.45, 1}, {$18.80, 15}, {$16.82, 20}, {$22...|
|   6|   1|[{$433.47, 5}, {$34.39, 10}, {$34.96, 12}, {$-4...|
|   8|   1|[{$32.67, 2}, {$16.20, 23}, {$18.71, 27}, {$47....|
|   8|   3|[{$21.24, 5}, {$19.27, 12}, {$122.29, 14}, {$13...|
|  13|   0|[{$1128.96, 18}, {$16.34, 22}, {$314.23, 7}, {$...|
|  13|   1|[{$16.81, 16}, {$43.04, 18}, {$17.67, 23}, {

А что с этим делать дальше? Можно перейти к RDD и построчно обрабатывать

In [148]:
row = trans_data_simple_seq.rdd.take(1)

In [149]:
row

[Row(User=0, Card=1, sequence=[Row(Amount='$20.61', Day=29), Row(Amount='$42.80', Day=6), Row(Amount='$13.49', Day=6), Row(Amount='$149.22', Day=17), Row(Amount='$172.21', Day=1), Row(Amount='$58.68', Day=5), Row(Amount='$33.22', Day=25), Row(Amount='$154.21', Day=29), Row(Amount='$10.47', Day=9), Row(Amount='$6.84', Day=4), Row(Amount='$87.28', Day=1), Row(Amount='$81.00', Day=15), Row(Amount='$-81.00', Day=15), Row(Amount='$39.80', Day=25), Row(Amount='$5.22', Day=1), Row(Amount='$14.08', Day=2), Row(Amount='$30.24', Day=13), Row(Amount='$66.30', Day=16), Row(Amount='$12.31', Day=24), Row(Amount='$120.10', Day=15), Row(Amount='$38.08', Day=1), Row(Amount='$63.82', Day=31), Row(Amount='$151.48', Day=6), Row(Amount='$50.01', Day=15), Row(Amount='$45.55', Day=24), Row(Amount='$103.04', Day=2), Row(Amount='$12.95', Day=13), Row(Amount='$10.26', Day=12), Row(Amount='$65.00', Day=13), Row(Amount='$1.49', Day=22), Row(Amount='$1.51', Day=10), Row(Amount='$24.14', Day=17), Row(Amount='$28.64

In [150]:
row[0]["sequence"]

[Row(Amount='$20.61', Day=29),
 Row(Amount='$42.80', Day=6),
 Row(Amount='$13.49', Day=6),
 Row(Amount='$149.22', Day=17),
 Row(Amount='$172.21', Day=1),
 Row(Amount='$58.68', Day=5),
 Row(Amount='$33.22', Day=25),
 Row(Amount='$154.21', Day=29),
 Row(Amount='$10.47', Day=9),
 Row(Amount='$6.84', Day=4),
 Row(Amount='$87.28', Day=1),
 Row(Amount='$81.00', Day=15),
 Row(Amount='$-81.00', Day=15),
 Row(Amount='$39.80', Day=25),
 Row(Amount='$5.22', Day=1),
 Row(Amount='$14.08', Day=2),
 Row(Amount='$30.24', Day=13),
 Row(Amount='$66.30', Day=16),
 Row(Amount='$12.31', Day=24),
 Row(Amount='$120.10', Day=15),
 Row(Amount='$38.08', Day=1),
 Row(Amount='$63.82', Day=31),
 Row(Amount='$151.48', Day=6),
 Row(Amount='$50.01', Day=15),
 Row(Amount='$45.55', Day=24),
 Row(Amount='$103.04', Day=2),
 Row(Amount='$12.95', Day=13),
 Row(Amount='$10.26', Day=12),
 Row(Amount='$65.00', Day=13),
 Row(Amount='$1.49', Day=22),
 Row(Amount='$1.51', Day=10),
 Row(Amount='$24.14', Day=17),
 Row(Amount='$28.